# Fact Set Inventory — Gold Layer

Central fact table at the grain of one row per set + part + color. Joins the full chain from sets through inventories to inventory parts.

## What this notebook does

1. **Read** — Loads sets, inventories, inventory parts, parts, colors, and theme hierarchy from silver/gold.
2. **Transform** — Joins the full chain: sets → inventories → inventory_parts, enriched with part, color, and theme names.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/gold/delta_volume/fct_set_inventory`.
4. **Register** — Creates the catalog table `lego_brickbase.gold.fct_set_inventory`.

## Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col


SILVER_SETS            = "lego_brickbase.silver.foundation_sets"
SILVER_INVENTORIES     = "lego_brickbase.silver.foundation_inventories"
SILVER_INVENTORY_PARTS = "lego_brickbase.silver.foundation_inventory_parts"
SILVER_PARTS           = "lego_brickbase.silver.foundation_parts"
SILVER_COLORS          = "lego_brickbase.silver.foundation_colors"
SILVER_PART_CATEGORIES = "lego_brickbase.silver.foundation_part_categories"
GOLD_THEME_HIERARCHY   = "lego_brickbase.gold.dim_theme_hierarchy"
DELTA_TABLE_PATH       = "/Volumes/lego_brickbase/gold/delta_volume/fct_set_inventory"
CATALOG_TABLE          = "lego_brickbase.gold.fct_set_inventory"

## Read and Transform

Join the full inventory chain and enrich with descriptive attributes from dimension tables.

In [ ]:
df_sets       = spark.table(SILVER_SETS)
df_inv        = spark.table(SILVER_INVENTORIES)
df_inv_parts  = spark.table(SILVER_INVENTORY_PARTS)
df_parts      = spark.table(SILVER_PARTS)
df_colors     = spark.table(SILVER_COLORS)
df_part_cats  = spark.table(SILVER_PART_CATEGORIES)
df_themes     = spark.table(GOLD_THEME_HIERARCHY)

df_fct = (
    df_inv_parts
    .join(df_inv, df_inv_parts["inventory_id"] == df_inv["inventory_id"], "inner")
    .join(df_sets, df_inv["set_key"] == df_sets["set_key"], "inner")
    .join(df_parts, df_inv_parts["part_key"] == df_parts["part_key"], "left")
    .join(df_colors, df_inv_parts["color_key"] == df_colors["color_key"], "left")
    .join(df_part_cats, df_parts["part_category_key"] == df_part_cats["part_category_key"], "left")
    .join(df_themes, df_sets["theme_key"] == df_themes["theme_key"], "left")
    .select(
        df_sets["set_key"],
        df_inv_parts["part_key"],
        df_inv_parts["color_key"],
        df_sets["set_number"],
        df_sets["set_name"],
        df_sets["year"],
        df_themes["theme_name"],
        df_themes["root_theme_name"],
        df_inv["inventory_id"],
        df_parts["part_name"],
        df_part_cats["part_category_name"],
        df_colors["color_name"],
        df_colors["rgb"].alias("color_rgb"),
        df_colors["is_transperent"].alias("is_transparent_color"),
        df_inv_parts["quantity"],
        df_inv_parts["is_spare"],
        df_inv_parts["image_url"],
    )
)

df_fct.printSchema()
df_fct.display(10, truncate=False)

## Write to Delta Volume, Register Catalog Table, and Apply Constraints

In [ ]:
(
    df_fct
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

# CREATE TABLE with inline PK/FK constraints (informational, not enforced)
# PK columns must be declared NOT NULL per Databricks Unity Catalog requirements
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE} (
        set_key              STRING  NOT NULL COMMENT 'Foreign key to dim_set; identifies the parent set',
        part_key             STRING  NOT NULL COMMENT 'Foreign key to dim_part; identifies the LEGO part',
        color_key            INTEGER NOT NULL COMMENT 'Foreign key to dim_color; identifies the color of the part in this inventory line',
        set_number           STRING           COMMENT 'Official LEGO set number (e.g. 75192-1)',
        set_name             STRING           COMMENT 'Official name of the LEGO set',
        year                 INTEGER          COMMENT 'Year the set was released',
        theme_name           STRING           COMMENT 'Name of the direct theme the set belongs to',
        root_theme_name      STRING           COMMENT 'Name of the top-level ancestor theme',
        inventory_id         INTEGER NOT NULL COMMENT 'Identifier of the specific inventory version this line belongs to',
        part_name            STRING           COMMENT 'Descriptive name of the LEGO part',
        part_category_name   STRING           COMMENT 'Category the part belongs to (e.g. Brick, Technic)',
        color_name           STRING           COMMENT 'Human-readable name of the part color',
        color_rgb            STRING           COMMENT 'Six-digit hexadecimal RGB value of the part color',
        is_transparent_color BOOLEAN          COMMENT 'True if the part color is a transparent variant',
        quantity             INTEGER          COMMENT 'Number of times this part in this color appears in the inventory',
        is_spare             BOOLEAN          COMMENT 'True if this part is included as a spare (extra) piece',
        image_url            STRING           COMMENT 'URL to the image of this specific part in this color',
        CONSTRAINT fct_set_inventory_pk
            PRIMARY KEY (inventory_id, part_key, color_key),
        CONSTRAINT fct_set_inventory_fk_set
            FOREIGN KEY (set_key)   REFERENCES lego_brickbase.gold.dim_set   (set_key),
        CONSTRAINT fct_set_inventory_fk_part
            FOREIGN KEY (part_key)  REFERENCES lego_brickbase.gold.dim_part  (part_key),
        CONSTRAINT fct_set_inventory_fk_color
            FOREIGN KEY (color_key) REFERENCES lego_brickbase.gold.dim_color (color_key)
    )
    COMMENT 'Central fact table at the grain of one row per inventory + part + color. Joins the full chain from sets through inventories to inventory parts, enriched with part, color, and theme descriptors.'
""")

spark.sql(f"""
    INSERT INTO {CATALOG_TABLE}
    SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")

print(f"Catalog table ready with constraints: {CATALOG_TABLE}")